In [6]:
import requests
import pandas as pd
import re
import time
from typing import Dict, List, Any

In [7]:
def clean_title(title: str) -> str:
    t = title

    # 1. 괄호, 대괄호 안의 내용 싹 다 날림
    t = re.sub(r'[\(\[].*?[\)\]]', '', t)

    # 2. 하이픈(-), 슬래시(/), 콜론(:) 뒤에 나오는 부제/굿즈 완벽 차단
    t = re.sub(r'\s+[-/:]\s+.*$', '', t)

    # 3. "1~3", "13-14" 같은 범위형 숫자 날림
    t = re.sub(r'\s*\d+\s*[~-]\s*\d+.*$', '', t)

    # 4. 앞선 필터를 뚫고 나온 특전/굿즈 관련 키워드 날림 (초판, 아크릴 등 추가)
    t = re.sub(r'\s*(\+|특전|세트|단행본|합본|박스|패키지|특별판|한정판|부록|에디션|초판|아크릴|스티커|포스터).*$', '', t)

    # 5. 문장 맨 끝에 덩그러니 남은 " 19", " 4", "시즌 2" 같은 권수/시즌 날림
    t = re.sub(r'\s*(시즌|part)?\s*\d+\s*(권|부|화|기)?\s*$', '', t, flags=re.IGNORECASE)

    return t.strip()

In [8]:
def assign_genres_by_score(title: str, desc: str) -> str:
    text = f"{title} {desc}".lower()

    genre_keywords = {
        '로맨스': ['로맨스', '연애', '설렘', '결혼', '첫사랑', '짝사랑', '사내연애'],
        '판타지': ['판타지', '던전', '헌터', '몬스터', '각성', '스킬', '마법', '게이트', '정령', '이세계',
                '로판', '황태자', '악녀', '영애', '공작', '계약결혼', '황제',
                '회귀', '빙의', '환생', '전생', '시스템', '상태창', '멸망'],
        '무협/사극': ['무협', '강호', '교주', '무공', '검술', '천마', '무림', '조선', '왕비', '궁중'],
        '액션': ['액션', '격투', '싸움', '학원액션', '히어로'],
        '스릴러/범죄': ['스릴러', '범죄', '미스터리', '형사', '추적', '사건', '추리', '경찰', '살인',
                  '느와르', '조폭', '갱스터', '주먹', '복수', '일진'],
        '공포/호러': ['공포', '괴담', '귀신', '악령', '좀비', '오컬트', '저주'],
        'SF/미래': ['sf', '우주', '미래', '안드로이드', '로봇', '사이버펑크', '가상현실'],
        '코미디/힐링': ['일상', '코믹', '개그', '가족', '요리', '음식', '동물', '고양이', '강아지', '힐링', '코미디'],
        '드라마/성장': ['드라마', '직장인', '학교', '학생', '청춘', '성장', '우정'],
        '음악/스포츠': ['스포츠', '축구', '야구', '농구', '아이돌', '음악', '밴드', '무대', '데뷔']
    }

    scores = {genre: 0 for genre in genre_keywords}
    for genre, keywords in genre_keywords.items():
        for kw in keywords:
            if kw in text:
                scores[genre] += 2 if kw in ['회귀', '빙의', '무공', '황태자', '던전', '좀비', '로판', '시스템', '느와르'] else 1

    sorted_genres = sorted([(g, s) for g, s in scores.items() if s > 0], key=lambda x: x[1], reverse=True)
    return ", ".join([g[0] for g in sorted_genres[:2]]) if sorted_genres else '드라마/성장'

In [9]:
class AladdinCollector:
    def __init__(self, ttb_key: str):
        self.ttb_key = ttb_key
        self.base_url = "http://www.aladin.co.kr/ttb/api/ItemList.aspx"

    def fetch_all_webtoons_multi_query(self, max_pages_per_query: int = 20) -> List[Dict[str, Any]]:
        raw_books = []
        seen_ids = set()

        query_types = ['Bestseller', 'ItemNewAll', 'ItemNewSpecial']

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }

        print("알라딘 웹툰 데이터 대량 수집을 시작합니다...")

        for q_type in query_types:
            print(f"\n탐색 모드: {q_type} (최대 {max_pages_per_query}페이지)")
            for page in range(1, max_pages_per_query + 1):
                params = {
                    'ttbkey': self.ttb_key,
                    'QueryType': q_type,
                    'SearchTarget': 'Book',
                    'CategoryId': 7443,
                    'output': 'js',
                    'Version': '20131101',
                    'MaxResults': 50,
                    'Start': page,
                    'OptResult': 'ratingInfo'
                }
                try:
                    res = requests.get(self.base_url, params=params, headers=headers)
                    res.raise_for_status()
                    data = res.json()

                    items = data.get('item', [])
                    if not items:
                        break

                    for item in items:
                        item_id = item.get('itemId')
                        if item_id and item_id not in seen_ids:
                            seen_ids.add(item_id)
                            raw_books.append(item)

                    print(f"  ... {page}페이지 완료 (현재 고유 데이터 누적: {len(raw_books)}개)")
                    time.sleep(0.3)

                except Exception as e:
                    print(f"{q_type} - {page}페이지 수집 중 에러 발생: {e}")
                    break

        return raw_books

In [ ]:
if __name__ == "__main__":
    ALADDIN_TTB_KEY = "" # API 키 입력

    collector = AladdinCollector(ttb_key=ALADDIN_TTB_KEY)

    raw_data = collector.fetch_all_webtoons_multi_query(max_pages_per_query=20)

    if raw_data:
        processed_list = []
        for item in raw_data:
            full_title = item.get('title', '')
            series_name = clean_title(full_title)
            desc = item.get('description', '')

            item_id = item.get('itemId', '')
            clean_link = f"https://www.aladin.co.kr/shop/wproduct.aspx?ItemId={item_id}" if item_id else ""

            processed_list.append({
                'series_title': series_name,
                'original_title': full_title,
                'author': item.get('author', ''),
                'publisher': item.get('publisher', ''),
                'sales_point': int(item.get('salesPoint', 0)),
                'description': desc,
                'cover_url': item.get('cover', ''),
                'clean_link': clean_link,
                'pub_date': item.get('pubDate', '')
            })

        df_raw = pd.DataFrame(processed_list)
        print(f"\n1차 원본 수집 완료: 총 {len(df_raw)}개의 고유 단행본 확보!")

        agg_funcs = {
            'original_title': lambda x: f"총 {len(x)}권 포함 (대표작: {list(x)[0]})",
            'author': 'first',
            'publisher': 'first',
            'sales_point': 'max',
            'description': 'first',
            'cover_url': 'first',
            'clean_link': 'first',
            'pub_date': 'max'
        }

        print("완벽한 이름 기준으로 시리즈 병합 및 장르 태깅을 진행합니다...")
        df_series = df_raw.groupby('series_title').agg(agg_funcs).reset_index()

        df_series['genres'] = df_series.apply(lambda row: assign_genres_by_score(row['series_title'], row['description']), axis=1)
        df_series = df_series.rename(columns={'clean_link': 'link'})

        final_cols = ['series_title', 'genres', 'author', 'sales_point', 'description', 'publisher', 'pub_date', 'cover_url', 'link', 'original_title']
        df_final = df_series[final_cols].sort_values(by='sales_point', ascending=False).reset_index(drop=True)

        print(f"최종 정제 완료: 중복 없이 압축된 총 {len(df_final)}개의 고유 웹툰 시리즈 추출 성공!")

        csv_filename = "aladin_webtoon_data.csv"
        df_final.to_csv(csv_filename, index=False, encoding="utf-8-sig")
        print(f"파일 저장 완료: {csv_filename}")

    else:
        print("데이터를 수집하지 못했습니다.")

알라딘 웹툰 데이터 대량 수집을 시작합니다...

탐색 모드: Bestseller (최대 20페이지)
  ... 1페이지 완료 (현재 고유 데이터 누적: 50개)
  ... 2페이지 완료 (현재 고유 데이터 누적: 100개)
  ... 3페이지 완료 (현재 고유 데이터 누적: 150개)
  ... 4페이지 완료 (현재 고유 데이터 누적: 200개)
  ... 5페이지 완료 (현재 고유 데이터 누적: 250개)
  ... 6페이지 완료 (현재 고유 데이터 누적: 300개)
  ... 7페이지 완료 (현재 고유 데이터 누적: 350개)
  ... 8페이지 완료 (현재 고유 데이터 누적: 400개)
  ... 9페이지 완료 (현재 고유 데이터 누적: 450개)
  ... 10페이지 완료 (현재 고유 데이터 누적: 500개)
  ... 11페이지 완료 (현재 고유 데이터 누적: 550개)
  ... 12페이지 완료 (현재 고유 데이터 누적: 600개)
  ... 13페이지 완료 (현재 고유 데이터 누적: 650개)
  ... 14페이지 완료 (현재 고유 데이터 누적: 700개)
  ... 15페이지 완료 (현재 고유 데이터 누적: 750개)
  ... 16페이지 완료 (현재 고유 데이터 누적: 800개)
  ... 17페이지 완료 (현재 고유 데이터 누적: 850개)
  ... 18페이지 완료 (현재 고유 데이터 누적: 900개)
  ... 19페이지 완료 (현재 고유 데이터 누적: 915개)

탐색 모드: ItemNewAll (최대 20페이지)
  ... 1페이지 완료 (현재 고유 데이터 누적: 925개)
  ... 2페이지 완료 (현재 고유 데이터 누적: 934개)
  ... 3페이지 완료 (현재 고유 데이터 누적: 946개)

탐색 모드: ItemNewSpecial (최대 20페이지)
  ... 1페이지 완료 (현재 고유 데이터 누적: 946개)

1차 원본 수집 완료: 총 946개의 고유 단행본 확보!
완벽한 이름 기준으로 시리즈 병합 및 장르 태깅을 진행합니